# Model Accuracy Report

This notebook evaluates and compares LSTM and Prophet price forecasts against actual historical prices stored in Supabase.

In [29]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Locate the backend folder reliably even when the notebook server cwd is the workspace root.
def find_backend_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "backend",
        cwd.parent,
        cwd.parent / "backend",
        cwd / "app",
    ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "app" / "core" / "config.py").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the backend root. "
        "Run this notebook from the repository workspace or open it from backend/evaluation/ with the backend source available."
    )

BACKEND_ROOT = find_backend_root()
sys.path.append(str(BACKEND_ROOT))

env_file = BACKEND_ROOT / ".env" / ".env.production"
if not env_file.exists():
    env_file = BACKEND_ROOT / ".env" / ".env.development"
    print("Using development env file:", env_file)
else:
    print("Using production env file:", env_file)

os.environ.setdefault("ENV_FILE", str(env_file))

from app.core.config import settings
from app.core.database.supabase import supabase

print("Connecting to Supabase using backend config...")
print("ENV_FILE:", os.environ.get("ENV_FILE"))
print("SUPABASE_URL:", settings.SUPABASE_URL)

try:
    # In v2.x, if this line doesn't throw an exception, the connection is solid!
    test = supabase.table("items").select("id").limit(1).execute()
    
    # Check if we actually got a data list back
    if hasattr(test, 'data'):
        print("✅ Supabase connected successfully! Database responded cleanly.")
        print(f"Sample data check: {test.data}")
except Exception as e:
    raise RuntimeError(
        "Supabase direct connection failed. Ensure your .env file contains valid SUPABASE_URL and SUPABASE_KEY.\n"
        f"Original error: {e}"
    )

Using production env file: D:\Github\AOL-AI\backend\.env\.env.production
Connecting to Supabase using backend config...
ENV_FILE: D:\Github\AOL-AI\backend\.env\.env.production
SUPABASE_URL: https://mcwojgalcdeztztzybcu.supabase.co
✅ Supabase connected successfully! Database responded cleanly.
Sample data check: [{'id': '60d0a128-9783-40c9-8baf-6f2469dd7dea'}]


## Fetch Historical Prices and Model Predictions

Fetch actual historical prices and both model predictions from Supabase tables. Replace placeholder table and column names with your schema.

In [30]:
def fetch_table(table_name: str, columns: list[str]):
    try:
        # In v2.x, this directly returns the data or throws an exception if it fails
        response = supabase.table(table_name).select(",".join(columns)).execute()
        return pd.DataFrame(response.data)
    except Exception as e:
        raise RuntimeError(f"Failed to fetch data from {table_name}. Original error: {e}")

# Your exact table and column definitions
actual_table = 'historical_prices'
predictions_table = 'ai_predictions'

actual_columns = ['id', 'item_id', 'date', 'median_price', 'volume']
pred_columns = ['id', 'item_id', 'target_date', 'predicted_price', 'model_used', 'created_at']

# Fetch dataframes using the updated function
print("Fetching tables from Supabase...")
actual_df = fetch_table(actual_table, actual_columns)
predictions_df = fetch_table(predictions_table, pred_columns)

# Normalize predictions: rename target_date -> date for easier joins
if 'target_date' in predictions_df.columns:
    predictions_df = predictions_df.rename(columns={'target_date': 'date'})
predictions_df['model_used'] = predictions_df['model_used'].astype(str)

# Split predictions by model_used values (case-insensitive)
lstm_df = predictions_df[predictions_df['model_used'].str.lower() == 'lstm_v1'].copy()
prophet_df = predictions_df[predictions_df['model_used'].str.lower() == 'prophet_v1'].copy()

print('\nSuccessfully pulled data!')
print('  actual_df:   ', len(actual_df))
print('  predictions: ', len(predictions_df))
print('  lstm_df:     ', len(lstm_df))
print('  prophet_df:  ', len(prophet_df))

Fetching tables from Supabase...

Successfully pulled data!
  actual_df:    1000
  predictions:  462
  lstm_df:      187
  prophet_df:   275


## Clean and Chronologically Align Data by Date

Convert dates to datetime, sort each dataframe, merge on the date index, and drop rows with missing values.

In [31]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Clean and isolate the historical baseline
actual_clean = prepare_df_robust(actual_filtered, 'date', 'median_price', 'actual_price')

# 2. Clean and evaluate LSTM individually
lstm_clean = prepare_df_robust(lstm_filtered, 'date', 'predicted_price', 'lstm_price')
lstm_merged = actual_clean.merge(lstm_clean, on='date', how='inner')

# 3. Clean and evaluate Prophet individually
prophet_clean = prepare_df_robust(prophet_filtered, 'date', 'predicted_price', 'prophet_price')
prophet_merged = actual_clean.merge(prophet_clean, on='date', how='inner')

print("=== INDEPENDENT EVALUATION COUNTS ===")
print(f"LSTM historical comparison pairs found: {len(lstm_merged)}")
print(f"Prophet historical comparison pairs found: {len(prophet_merged)}\n")

# 4. Custom MAPE calculation to prevent zero division errors
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=np.float64), np.array(y_pred, dtype=np.float64)
    mask = y_true != 0
    if not np.any(mask) or len(y_true) == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# 5. Build individual evaluation metrics
metrics_summary = []

# Process LSTM metrics if data exists
if len(lstm_merged) > 0:
    act = lstm_merged['actual_price'].values
    prd = lstm_merged['lstm_price'].values
    metrics_summary.append({
        'Model': 'LSTM',
        'Sample Size': len(lstm_merged),
        'MAE': mean_absolute_error(act, prd),
        'RMSE': np.sqrt(mean_squared_error(act, prd)),
        'MAPE': calculate_mape(act, prd)
    })
else:
    metrics_summary.append({'Model': 'LSTM', 'Sample Size': 0, 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan})

# Process Prophet metrics if data exists
if len(prophet_merged) > 0:
    act = prophet_merged['actual_price'].values
    prd = prophet_merged['prophet_price'].values
    metrics_summary.append({
        'Model': 'Prophet',
        'Sample Size': len(prophet_merged),
        'MAE': mean_absolute_error(act, prd),
        'RMSE': np.sqrt(mean_squared_error(act, prd)),
        'MAPE': calculate_mape(act, prd)
    })
else:
    metrics_summary.append({'Model': 'Prophet', 'Sample Size': 0, 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan})

# 6. Format and render a clean summary table
metrics_df = pd.DataFrame(metrics_summary)

for col in ['MAE', 'RMSE']:
    metrics_df[col] = metrics_df[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A (Future Predictions Only)")
metrics_df['MAPE'] = metrics_df['MAPE'].map(lambda x: f"{x:.2f}%" if pd.notna(x) else "N/A")

print("Model Accuracy Comparison Report")
print("--------------------------------")
metrics_df

=== INDEPENDENT EVALUATION COUNTS ===
LSTM historical comparison pairs found: 0
Prophet historical comparison pairs found: 4

Model Accuracy Comparison Report
--------------------------------


,Model,Sample Size,MAE,RMSE,MAPE
0,LSTM,0,N/A (Future Predictions Only),N/A (Future Predictions Only),N/A
1,Prophet,4,77.1325,82.9253,172.44%


## Print Comparison Summary Table

Display the evaluation metrics side by side for both models.

In [32]:
print('Model Accuracy Comparison')
print('--------------------------')
print(metrics_df.to_string(index=False))

Model Accuracy Comparison
--------------------------
  Model  Sample Size                           MAE                          RMSE    MAPE
   LSTM            0 N/A (Future Predictions Only) N/A (Future Predictions Only)     N/A
Prophet            4                       77.1325                       82.9253 172.44%


## Metric Formula Summary

- **Mean Absolute Error (MAE)** measures the average magnitude of errors in a set of predictions, without considering their direction.
- **Root Mean Squared Error (RMSE)** measures the square root of the average squared differences between predicted and actual values, giving higher weight to larger errors.
- **Mean Absolute Percentage Error (MAPE)** measures the average absolute percent difference between predicted and actual values.

### Formulas

- MAE:  
  \ ( \text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| \ )

- RMSE:  
  \ ( \text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} \ )

- MAPE:  
  \ ( \text{MAPE} = \frac{100}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right| \ )

Replace the placeholder table names and columns with your actual Supabase schema before running the notebook.